# 28 · Relate a table against a pandas merge

**Question.** Snow's Table XII says which water company supplied each of 32 sub-districts. Does
bringing that company onto the 64 rows of his Table VIII land the same category on the same row
as an independent left join — and does it do the right thing where a join does the wrong thing?

**There is no arithmetic here, and that is the point.** A relate brings a *category* across; no
number is computed. So this is not a check on a sum. It is a check on the **matching**, and on
one case where the two implementations are meant to disagree.

`pandas.DataFrame.merge` is the operation a practitioner would otherwise reach for, written by
other authors. Where it agrees, the agreement is worth having. Where it does not, the difference
is the finding.


In [ ]:
# Only what this notebook imports. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested:", NEEDED)
except ImportError:
    print("Not Pyodide; using the environment as installed.")


## Parity

**Rebuilt 2026-10-28 for `relate_table` 0.3.0.** Eight cases now, recomputed from the fixture's
three input tables and each case's own request. Three things about the widget changed and the old
fixture described none of them: keys are compared with punctuation and case **folded** by default,
a key the right table does not carry **refuses** until the practitioner declares what to do with
it, and the **retained share** of each numeric variable is part of the result.

The folding is reimplemented in `checks/check_28_relate_table.py` from its declared rule and never
imported. Borrowing Fieldwork's own `normaliseName` would compare the widget with itself.


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_28_relate_table import run

report = run(Path.cwd() / "fixtures" / "relate-snow-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{c['note']}")
    print(f"  compared {c['comparison']}, bringing {c['bring']}"
          f"{', declared ' + c['decision'] if c['decision'] else ''}")
    print(f"  keys matched   Fieldwork {c['fieldwork']['matchedKeys']:3d}   pandas {c['pandas']['matchedKeys']:3d}")
    print(f"  rows           {c['fieldwork']['rowCount']:12d}   {c['pandas']['rowCount']:9d}")
    print(f"  dropped        {c['fieldwork']['dropped']:12d}   {c['pandas']['dropped']:9d}")
    print(f"  kept           {(c['fieldwork']['kept'] or 0):12d}   {c['pandas']['kept']:9d}")
    for s in c["fieldwork"]["shares"]:
        mine = c["pandas"]["shares"][s["measure"]]
        pct = "-" if s["share"] is None else f"{s['share']*100:.2f}%"
        ours = "-" if mine["share"] is None else f"{mine['share']*100:.2f}%"
        print(f"  share {s['measure']:<20} {pct:>8}   {ours:>8}")
    for name, ok in c["checks"].items():
        print(f"    {'OK ' if ok else 'DIFF'} {name}")


## The cases worth the check: two refusals

Fieldwork refuses twice where a merge proceeds, and each refusal is the behaviour under check
rather than a limitation. A **right table giving one key two different values** is refused outright;
a merge multiplies the rows, so both companies appear in the stratification carrying the same
population and no total reveals it. A **key the right table does not carry, with nothing declared**
about it, is also refused from 0.3.0; a merge proceeds and leaves NaN.

What a merge does instead is recorded beside each, because that is what the refusal buys.


In [ ]:
for r in report["refusals"]:
    print(r["note"])
    if r.get("constructed"):
        print("  constructed:", r["constructed"])
    print("  Fieldwork refuses:")
    print("   ", r["fieldworkRefused"])
    print("  a merge instead:")
    print("   ", r["whatAMergeDoesInstead"])
    for name, ok in r["checks"].items():
        print(f"    {'OK ' if ok else 'DIFF'} {name}")
    print()


## The pair this check could not reach before

The previous version of this notebook recorded, under what it could not establish: *"St. George,
Camberwell" against "St George Camberwell" is the case a practitioner will actually meet, and
neither this fixture nor this check reaches it.*

It is reachable now. The published sub-district boundary is a second source for the same 32 names
and spells nine of them without their comma. The fixture carries the same two tables compared both
ways, so the two cases differ in one knob and nothing else.

**And the shares are where it matters.** The two variables do not fall in step, so excluding the
nine unmatched sub-districts moves the mortality rate rather than only its precision. The identity
Fieldwork states — that an inner join multiplies a rate of one variable over another by exactly the
ratio of their retained shares — is recomputed here from the frames and the shares separately.


In [ ]:
pairs = [c for c in report["comparisons"] if c["bring"] == "supplier"]
for c in pairs:
    f, p = c["fieldwork"], c["pandas"]
    print(f"compared {c['comparison']:5}  keys {f['matchedKeys']:2}/32  rows {f['rowCount']:2}  dropped {f['dropped']:2}")
    for s in f["shares"]:
        print(f"    {s['measure']:<20} {s['related']:>9,.0f} of {s['all']:>9,.0f}"
              f"  = {(0 if s['share'] is None else s['share'])*100:6.2f}%")
print()
i = report["shareIdentity"]
print(i["claim"])
print(f"  rate over every offered row : {i['ratePerThousandAllRows']:.3f} per 1000")
print(f"  multiplier from the shares  : {i['multiplierFromShares']:.4f}")
print(f"  predicted                   : {i['ratePerThousandPredicted']:.3f} per 1000")
print(f"  observed                    : {i['ratePerThousandMatchedRows']:.3f} per 1000")
print(f"  holds                       : {i['holds']}")
print()
print(i["reading"])


## What this cannot establish

**That folding is the right matching.** The cases above show what each setting does to two
published spellings of the same 32 sub-districts. They do not show that nine names differing by a
comma are nine names of the same place — that is a judgement about the sources, and a fold that
joined two genuinely different places would be wrong in exactly the way this check cannot see.

**That the practitioner's declaration is the right one.** The check confirms that `drop` and `keep`
produce the totals each claims. Which a study should use is not a question a merge can answer.


In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas  :", report["environment"]["pandas"])
print("runtime :", report["environment"]["runtime"])
